# 04.5 · Scaling features: who needs it and who does not

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/04-trustworthy-models/05-scaling/lesson.ipynb)

Read the [README](README.md) first. This notebook:

1. standardises and min-max scales five numbers by hand and with scikit-learn,
2. shows how unscaled distances let `spend` drown out `orders`,
3. measures that on the real churn table: unscaled KNN is secretly a one-feature model,
4. compares models that care about scale (KNN, SVM, logistic regression) with one that does not (a tree),
5. applies the training mean and standard deviation to a brand-new customer.

In [1]:
# --- data helper: works locally and on Colab -------------------------------
from pathlib import Path
import urllib.request

def data_path(name):
    """Return a local path to data/<name>, downloading from GitHub on Colab."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data" / name
        if candidate.exists():
            return candidate
    url = f"https://raw.githubusercontent.com/cloudops-one/training-ai-ml/main/data/{name}"
    dest = Path(name)
    if not dest.exists():
        urllib.request.urlretrieve(url, dest)
    return dest

import sys, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
sys.path.insert(0, str(data_path("download.py").parent))
from download import load_retail

raw = load_retail()                                   # ~1 s after the first run
sales = raw[~raw["Invoice"].astype(str).str.startswith("C")]
sales = sales[(sales["Quantity"] > 0) & (sales["Price"] > 0)].dropna(subset=["Customer ID"]).drop_duplicates()
sales = sales.assign(Revenue=sales["Quantity"] * sales["Price"])

def customer_table(cutoff, months=3):
    """One row per customer: features from before `cutoff`, churned = no order in the next `months`."""
    cutoff = pd.Timestamp(cutoff)
    past = sales[sales["InvoiceDate"] < cutoff]
    future = sales[(sales["InvoiceDate"] >= cutoff) & (sales["InvoiceDate"] < cutoff + pd.DateOffset(months=months))]
    c = past.groupby("Customer ID").agg(orders=("Invoice", "nunique"), spend=("Revenue", "sum"),
                                        products=("StockCode", "nunique"), country=("Country", "first"),
                                        first_order=("InvoiceDate", "min"), last_order=("InvoiceDate", "max"))
    c["tenure_days"] = (cutoff - c["first_order"]).dt.days
    c["days_since_last"] = (cutoff - c["last_order"]).dt.days
    came_back = future.groupby("Customer ID")["Revenue"].sum().reindex(c.index).fillna(0) > 0
    c["churned"] = (~came_back).astype(int)
    return c.drop(columns=["first_order", "last_order"])

features = ["orders", "spend", "products", "tenure_days", "days_since_last"]
cust = customer_table("2011-09-01")
print(f"{len(cust):,} customers, churn rate {cust['churned'].mean():.1%}")

3,317 customers, churn rate 43.6%


## 1. By hand: five spends

In [2]:
spend5 = np.array([200, 400, 600, 800, 1000], dtype=float)

mu, sigma = spend5.mean(), spend5.std()          # NumPy's std() divides by n, like StandardScaler
print(f"mean = {mu}, std (÷n) = {sigma:.2f}")
print("z-scores:     ", ((spend5 - mu) / sigma).round(3))
print("min-max 0..1: ", (spend5 - spend5.min()) / (spend5.max() - spend5.min()))

mean = 600.0, std (÷n) = 282.84
z-scores:      [-1.414 -0.707  0.     0.707  1.414]
min-max 0..1:  [0.   0.25 0.5  0.75 1.  ]


In [3]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler
col = spend5.reshape(-1, 1)
print("StandardScaler:", StandardScaler().fit_transform(col).ravel().round(3))
print("MinMaxScaler:  ", MinMaxScaler().fit_transform(col).ravel())
print(f"pandas .std() divides by n-1 instead: {pd.Series(spend5).std():.2f}")

StandardScaler: [-1.414 -0.707  0.     0.707  1.414]
MinMaxScaler:   [0.   0.25 0.5  0.75 1.  ]
pandas .std() divides by n-1 instead: 316.23


Same numbers by hand and by library. One trap: Pandas' `.std()` divides by $n-1$ (316.23), NumPy's and scikit-learn's by $n$ (282.84). Mixing them gives slightly different z-scores. Use the scaler and you never have to think about it.

## 2. Three customers: who is A's nearest neighbour?

In [4]:
abc = pd.DataFrame({"orders": [2, 12, 2], "spend": [500, 520, 900]}, index=["A", "B", "C"])

from scipy.spatial.distance import cdist
print("unscaled distances from A:")
print(pd.Series(cdist(abc.loc[["A"]], abc)[0], index=abc.index).round(2).to_string())

unscaled distances from A:
A      0.00
B     22.36
C    400.00


Unscaled, B is A's near-twin (distance 22) and C is far away (400). But look at the customers: A and C both ordered twice; B ordered **six times as often**. The £20 vs £400 difference in spend shouts down the 2 vs 12 difference in orders because pounds are bigger numbers than orders, not because they matter more.

Divide each column by its standard deviation in the real churn table:

In [5]:
stds = cust[["orders", "spend"]].std()
print("std in the real table:", stds.round(1).to_dict())
abc_scaled = abc / stds
print("scaled distances from A:")
print(pd.Series(cdist(abc_scaled.loc[["A"]], abc_scaled)[0], index=abc.index).round(3).to_string())

std in the real table: {'orders': 5.6, 'spend': 5974.0}
scaled distances from A:
A    0.000
B    1.790
C    0.067


Now C is A's nearest neighbour (0.067) and B is far away (1.79). A difference of 10 orders is almost two "typical spreads" of orders; a difference of £400 is a small fraction of the typical spread of spend. Scaling made the distance measure every feature in its own units of "unusual".

## 3. The real churn table: unscaled KNN is a one-feature model

In [6]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    cust[features], cust["churned"], test_size=0.25, random_state=42, stratify=cust["churned"])
print(f"train {len(X_train):,} / test {len(X_test):,}")

from sklearn.neighbors import KNeighborsClassifier
rng = np.random.default_rng(0)
i, j = rng.integers(0, len(X_train), (2, 2000))                 # 2,000 random pairs of customers
diff2 = (X_train.values[i] - X_train.values[j]) ** 2
share = pd.Series(diff2.sum(axis=0) / diff2.sum(), index=features)
print("share of squared distance from each feature (unscaled):")
print(share.map("{:.2%}".format).to_string())

train 2,487 / test 830
share of squared distance from each feature (unscaled):
orders              0.00%
spend              99.93%
products            0.02%
tenure_days         0.02%
days_since_last     0.02%


In [7]:
sc = StandardScaler().fit(X_train)                            # learned from TRAIN rows only
Xs_train, Xs_test = sc.transform(X_train), sc.transform(X_test)

rows = []
for k in [5, 15, 51]:
    rows.append({"k": k,
                 "unscaled, 5 features": KNeighborsClassifier(k).fit(X_train, y_train).score(X_test, y_test),
                 "spend only": KNeighborsClassifier(k).fit(X_train[["spend"]], y_train).score(X_test[["spend"]], y_test),
                 "scaled, 5 features": KNeighborsClassifier(k).fit(Xs_train, y_train).score(Xs_test, y_test)})
pd.DataFrame(rows).set_index("k").round(3)

,"unscaled, 5 features",spend only,"scaled, 5 features"
k,,,
5,0.620,0.625,0.614
15,0.660,0.642,0.645
51,0.677,0.652,0.678


Summed over 2,000 random pairs of customers, **99.93%** of the squared distance comes from `spend`. The other four features only get a say between customers whose spend is almost identical. So the "five-feature" unscaled model behaves like a spend model with small tie-breakers: within about 2.5 points of the model given *only* spend.

Scaling does **not** automatically raise accuracy here: at k = 5 and 15 the scaled model is slightly *worse*, at k = 51 they tie. Spend happens to be a good feature, so letting it dominate was a lucky accident. Scaling changes *which features the model can hear*; whether that helps depends on the features (exercise 4 tries a better preparation). What it removes is an accident: without it, the units you happened to store the data in decide what the model pays attention to.

## 4. Which models care?

In [8]:
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

models = {"SVM (RBF kernel)": SVC(), "logistic regression": LogisticRegression(max_iter=10_000),
          "decision tree (depth 4)": DecisionTreeClassifier(max_depth=4, random_state=0)}
for name, m in models.items():
    raw_acc = m.fit(X_train, y_train).score(X_test, y_test)
    scaled_acc = m.fit(Xs_train, y_train).score(Xs_test, y_test)
    print(f"{name:<24} unscaled {raw_acc:.1%}   scaled {scaled_acc:.1%}")

SVM (RBF kernel)         unscaled 66.4%   scaled 68.3%
logistic regression      unscaled 67.2%   scaled 67.5%
decision tree (depth 4)  unscaled 65.1%   scaled 64.9%


In [9]:
lr_raw = LogisticRegression(max_iter=10_000).fit(X_train, y_train)
lr_scaled = LogisticRegression(max_iter=10_000).fit(Xs_train, y_train)
print(f"logistic regression, solver iterations: unscaled {lr_raw.n_iter_[0]}, scaled {lr_scaled.n_iter_[0]}")

logistic regression, solver iterations: unscaled 99, scaled 12


- **SVM** gains about 2 points: its RBF kernel is built on distances.
- **Logistic regression** ends up at a similar accuracy, but the solver needs **99** steps on unscaled data against **12** on scaled data, right at the edge of the default limit of 100 (one more feature or a few more customers and it stops early with a convergence warning). That is gradient descent from lesson 02.4: on badly scaled features the loss surface is a long narrow valley.
- **The tree** gives essentially the same accuracy either way. It asks "is spend above £X?"; dividing every spend by the same number moves the threshold but not which customers fall on each side. (Any difference in the last decimal comes from how ties between equal values are broken.)

## 5. Scaling a brand-new customer

At prediction time the model meets one customer at a time. They are scaled with the **training** mean and standard deviation, which the scaler stored when it was fitted.

In [10]:
new = pd.DataFrame([{"orders": 3, "spend": 900.0, "products": 40, "tenure_days": 200, "days_since_last": 30}])
print("training means:", dict(zip(features, sc.mean_.round(1))))
print("training stds: ", dict(zip(features, sc.scale_.round(1))))
print("new customer scaled:", sc.transform(new).round(2))
print("by hand, orders:", round((3 - sc.mean_[0]) / sc.scale_[0], 2))

training means: {'orders': np.float64(3.4), 'spend': np.float64(1510.1), 'products': np.float64(48.4), 'tenure_days': np.float64(176.1), 'days_since_last': np.float64(92.2)}
training stds:  {'orders': np.float64(5.2), 'spend': np.float64(5346.8), 'products': np.float64(61.7), 'tenure_days': np.float64(78.0), 'days_since_last': np.float64(77.4)}
new customer scaled: [[-0.07 -0.11 -0.14  0.31 -0.8 ]]
by hand, orders: -0.07


Never fit a scaler on the new data itself: one customer has no standard deviation, and a batch of new customers would be scaled differently from the training data the model learned on.

## Quiz answers

1. $z = (x - \mu)/\sigma$; the result has mean 0 and standard deviation 1. Min-max: $(x - \min)/(\max - \min)$, range 0 to 1.
2. Because distances add up squared differences; a feature measured in big units (pounds) dominates one in small units (orders), regardless of importance.
3. KNN, SVM, logistic/linear regression with regularisation, anything trained by gradient descent (including neural networks), PCA, K-Means. Not needed: decision trees, random forests, gradient-boosted trees.
4. On the training data only. The same fitted scaler (its stored mean and standard deviation) is then applied to validation, test and live data.
5. Min-max scaling squeezes everyone else into a tiny range (most customers end up near 0) because the range is set by the one extreme value. Standardising is less affected; a log transform first (04.3) is often better still.

**Next:** [04.6 Feature engineering](../06-feature-engineering/)